In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install idx2numpy

In [9]:
!pip install seqeval

In [10]:
!pip install pytorch-crf

In [11]:
%%bash
base_dir="/content/drive/MyDrive/DL-TH3/"
touch "$base_dir/phoner.py"
touch "$base_dir/uit_vsfc.py"
touch "$base_dir/train_phoner.py"
touch "$base_dir/lstm.py"
touch "$base_dir/train_vsfc.py"
touch "$base_dir/gru.py"
touch "$base_dir/train_vsfc_gru.py"
touch "$base_dir/phoner_dataset.py"
touch "$base_dir/bilstm_crf.py"
touch "$base_dir/train_phoner.py"

echo "Đã tạo xong"
ls -l "$base_dir"

Đã tạo xong
total 23737
-rw------- 1 root root 10819687 Nov 27 12:02 best_gru.pt
-rw------- 1 root root 13451377 Nov 27 12:01 best_lstm.pt
-rw------- 1 root root      971 Nov 27 12:22 bilstm_crf.py
-rw------- 1 root root      911 Nov 27 12:22 gru.py
-rw------- 1 root root      923 Nov 27 12:22 lstm.py
drwx------ 2 root root     4096 Nov 27 11:03 PhoNER_COVID19
-rw------- 1 root root     1340 Nov 27 12:22 phoner_dataset.py
-rw------- 1 root root        0 Nov 27 12:22 phoner.py
drwx------ 2 root root     4096 Nov 27 12:04 __pycache__
-rw------- 1 root root     3761 Nov 27 11:54 train-phoner.py
-rw------- 1 root root     3752 Nov 27 12:22 train_phoner.py
-rw------- 1 root root     3186 Nov 27 12:22 train_vsfc_gru.py
-rw------- 1 root root     3033 Nov 27 12:22 train_vsfc.py
drwx------ 2 root root     4096 Nov 26 15:43 UIT-VSFC
-rw------- 1 root root     2499 Nov 27 12:22 uit_vsfc.py


In [12]:
%%writefile "/content/drive/MyDrive/DL-TH3/uit_vsfc.py"
import torch
from torch.utils.data import Dataset
import json
import os
import string
from torch.nn import functional as F

def collate_fn(items: list, pad_idx=0) -> dict:
    input_ids = [item["input_ids"] for item in items]
    max_len = max([x.shape[0] for x in input_ids])

    input_ids = [
        F.pad(x, pad=(0, max_len - x.shape[0]), mode="constant", value=pad_idx).unsqueeze(0)
        for x in input_ids
    ]
    input_ids = torch.cat(input_ids, dim=0)

    label_ids = torch.tensor([item["label"] for item in items], dtype=torch.long)
    return {"input_ids": input_ids, "label_ids": label_ids}


class Vocab:
    def __init__(self, path: str):
        all_words = set()
        labels = set()
        for filename in os.listdir(path):
            if not filename.endswith(".json"):
                continue
            data = json.load(open(os.path.join(path, filename)))
            for item in data:
                sentence = self.preprocess_sentence(item["sentence"])
                all_words.update(sentence.split())
                labels.add(item["topic"])

        self.bos = "<s>"
        self.pad = "<p>"

        self.w2i = {word: idx for idx, word in enumerate(all_words, start=2)}
        self.w2i[self.pad] = 0
        self.w2i[self.bos] = 1
        self.i2w = {idx: word for word, idx in self.w2i.items()}

        self.l2i = {label: idx for idx, label in enumerate(labels)}
        self.i2l = {idx: label for label, idx in self.l2i.items()}

    def n_labels(self):
        return len(self.l2i)

    def __len__(self):
        return len(self.w2i)

    def preprocess_sentence(self, sentence: str) -> str:
        translator = str.maketrans("", "", string.punctuation)
        sentence = sentence.lower()
        sentence = sentence.translate(translator)
        return sentence


class VSFCDataset(Dataset):
    def __init__(self, json_path, vocab: Vocab):
        self.data = json.load(open(json_path))
        self.vocab = vocab

    def encode_sentence(self, sentence):
        sentence = self.vocab.preprocess_sentence(sentence)
        tokens = sentence.split()
        ids = [self.vocab.w2i.get(tok, 1) for tok in tokens]  # OOV=1
        return torch.tensor(ids, dtype=torch.long)

    def __getitem__(self, idx):
        item = self.data[idx]
        input_ids = self.encode_sentence(item["sentence"])
        label = self.vocab.l2i[item["topic"]]
        return {"input_ids": input_ids, "label": label}

    def __len__(self):
        return len(self.data)

Overwriting /content/drive/MyDrive/DL-TH3/uit_vsfc.py


# **Bài 1**

In [13]:
%%writefile "/content/drive/MyDrive/DL-TH3/lstm.py"
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, n_labels, hidden=256, num_layers=5,
                 bidirectional=False, pad_idx=0):
        super().__init__()
        self.hidden = hidden
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embed = nn.Embedding(vocab_size, 256, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=0.3 if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), n_labels)

    def forward(self, x):
        x = self.embed(x)
        output, (h, c) = self.lstm(x)
        last_hidden = h[-1]   # (B, H)
        logits = self.fc(last_hidden)
        return logits

Overwriting /content/drive/MyDrive/DL-TH3/lstm.py


In [14]:
%%writefile "/content/drive/MyDrive/DL-TH3/train_vsfc.py"
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

from uit_vsfc import Vocab, VSFCDataset, collate_fn
from lstm import LSTMClassifier

DATA_FOLDER = "/content/drive/MyDrive/DL-TH3/UIT-VSFC"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
vocab = Vocab(DATA_FOLDER)
train_ds = VSFCDataset(os.path.join(DATA_FOLDER, "UIT-VSFC-train.json"), vocab)
dev_ds   = VSFCDataset(os.path.join(DATA_FOLDER, "UIT-VSFC-dev.json"), vocab)
test_ds  = VSFCDataset(os.path.join(DATA_FOLDER, "UIT-VSFC-test.json"), vocab)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)

# Model
model = LSTMClassifier(
    vocab_size=len(vocab),
    n_labels=vocab.n_labels(),
    hidden=256,
    num_layers=5,
    bidirectional=False,
    pad_idx=0
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# early stopping
patience = 3
best_f1 = 0
bad_epochs = 0

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch in train_loader:
        x = batch["input_ids"].to(device)
        y = batch["label_ids"].to(device)

        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()   # ✔ phải cộng!

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} loss = {avg_loss:.4f}")

    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for batch in dev_loader:
            x = batch["input_ids"].to(device)
            y = batch["label_ids"].to(device)

            logits = model(x)
            pred = torch.argmax(logits, dim=-1)
            preds.extend(pred.tolist())
            golds.extend(y.tolist())

    f1 = f1_score(golds, preds, average="macro")
    print(f"Dev F1: {f1:.4f}")

    # early stopping
    if f1 > best_f1:
        best_f1 = f1
        bad_epochs = 0
        torch.save(model.state_dict(), "/content/drive/MyDrive/DL-TH3/best_lstm.pt")
        print("Saved best model.")
    else:
        bad_epochs += 1
        print(f"No improvement: {bad_epochs}/{patience}")

        if bad_epochs >= patience:
            print("Early stopping triggered!")
            break

# test
model.load_state_dict(torch.load("/content/drive/MyDrive/DL-TH3/best_lstm.pt"))
model.eval()

preds, golds = [], []
with torch.no_grad():
    for batch in test_loader:
        x = batch["input_ids"].to(device)
        y = batch["label_ids"].to(device)

        logits = model(x)
        pred = torch.argmax(logits, dim=-1)
        preds.extend(pred.tolist())
        golds.extend(y.tolist())

print("Test F1:", f1_score(golds, preds, average="macro"))


Overwriting /content/drive/MyDrive/DL-TH3/train_vsfc.py


In [15]:
!python "/content/drive/MyDrive/DL-TH3/train_vsfc.py"

Epoch 0 loss = 0.6943
Dev F1: 0.3752
Saved best model.
Epoch 1 loss = 0.5132
Dev F1: 0.5955
Saved best model.
Epoch 2 loss = 0.4568
Dev F1: 0.6043
Saved best model.
Epoch 3 loss = 0.4313
Dev F1: 0.6062
Saved best model.
Epoch 4 loss = 0.4034
Dev F1: 0.6053
No improvement: 1/3
Epoch 5 loss = 0.3803
Dev F1: 0.6152
Saved best model.
Epoch 6 loss = 0.3609
Dev F1: 0.6169
Saved best model.
Epoch 7 loss = 0.3438
Dev F1: 0.6227
Saved best model.
Epoch 8 loss = 0.3314
Dev F1: 0.6895
Saved best model.
Epoch 9 loss = 0.3246
Dev F1: 0.7080
Saved best model.
Epoch 10 loss = 0.3130
Dev F1: 0.7025
No improvement: 1/3
Epoch 11 loss = 0.2947
Dev F1: 0.7246
Saved best model.
Epoch 12 loss = 0.2922
Dev F1: 0.7281
Saved best model.
Epoch 13 loss = 0.2798
Dev F1: 0.7425
Saved best model.
Epoch 14 loss = 0.2687
Dev F1: 0.7477
Saved best model.
Epoch 15 loss = 0.2545
Dev F1: 0.7578
Saved best model.
Epoch 16 loss = 0.2476
Dev F1: 0.7413
No improvement: 1/3
Epoch 17 loss = 0.2423
Dev F1: 0.7560
No improvement

# **Bài 2**

In [16]:
%%writefile "/content/drive/MyDrive/DL-TH3/gru.py"
import torch
import torch.nn as nn

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, n_labels, hidden=256, num_layers=5, bidirectional=False, pad_idx=0):
        super().__init__()
        self.hidden = hidden
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embed = nn.Embedding(vocab_size, 256, padding_idx=pad_idx)

        self.gru = nn.GRU(
            input_size=256,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=0.3 if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), n_labels)

    def forward(self, x):
        x = self.embed(x)
        output, h = self.gru(x)
        last_hidden = h[-1]            # (B, hidden)
        logits = self.fc(last_hidden)
        return logits

Overwriting /content/drive/MyDrive/DL-TH3/gru.py


In [17]:
%%writefile "/content/drive/MyDrive/DL-TH3/train_vsfc_gru.py"
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

from uit_vsfc import Vocab, VSFCDataset, collate_fn
from gru import GRUClassifier

DATA_FOLDER = "/content/drive/MyDrive/DL-TH3/UIT-VSFC"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
vocab = Vocab(DATA_FOLDER)

train_ds = VSFCDataset(os.path.join(DATA_FOLDER, "UIT-VSFC-train.json"), vocab)
dev_ds   = VSFCDataset(os.path.join(DATA_FOLDER, "UIT-VSFC-dev.json"), vocab)
test_ds  = VSFCDataset(os.path.join(DATA_FOLDER, "UIT-VSFC-test.json"), vocab)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=lambda x: collate_fn(x, pad_idx=0))
dev_loader   = DataLoader(dev_ds, batch_size=16, shuffle=False, collate_fn=lambda x: collate_fn(x, pad_idx=0))
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=lambda x: collate_fn(x, pad_idx=0))

# Model
model = GRUClassifier(
    vocab_size=len(vocab),
    n_labels=vocab.n_labels(),
    hidden=256,
    num_layers=5,
    bidirectional=False,
    pad_idx=0
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Early stopping
best_f1 = 0
patience = 3
counter = 0

print("Start training...")

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch in train_loader:
        x = batch["input_ids"].to(device)
        y = batch["label_ids"].to(device)

        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} loss: {avg_loss:.4f}")

    model.eval()
    preds, golds = [], []

    with torch.no_grad():
        for batch in dev_loader:
            x = batch["input_ids"].to(device)
            y = batch["label_ids"].to(device)

            logits = model(x)
            pred = torch.argmax(logits, dim=-1)

            preds.extend(pred.tolist())
            golds.extend(y.tolist())

    f1 = f1_score(golds, preds, average="macro")
    print(f"Dev F1 = {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        counter = 0
        torch.save(model.state_dict(), "/content/drive/MyDrive/DL-TH3/best_gru.pt")
        print("Saved best model")
    else:
        counter += 1
        print(f"  Early stop counter = {counter}/{patience}")

        if counter >= patience:
            print(" Early stopping triggered!")
            break

    model.train()

model.load_state_dict(torch.load("/content/drive/MyDrive/DL-TH3/best_gru.pt"))
model.eval()

preds, golds = [], []
with torch.no_grad():
    for batch in test_loader:
        x = batch["input_ids"].to(device)
        y = batch["label_ids"].to(device)

        logits = model(x)
        pred = torch.argmax(logits, dim=-1)

        preds.extend(pred.tolist())
        golds.extend(y.tolist())

test_f1 = f1_score(golds, preds, average="macro")
print(" Results")
print(f"Best Dev F1 = {best_f1:.4f}")
print(f"Test F1     = {test_f1:.4f}")

Overwriting /content/drive/MyDrive/DL-TH3/train_vsfc_gru.py


In [18]:
!python "/content/drive/MyDrive/DL-TH3/train_vsfc_gru.py"

Start training...
Epoch 0 loss: 0.6510
Dev F1 = 0.5508
Saved best model
Epoch 1 loss: 0.4301
Dev F1 = 0.6172
Saved best model
Epoch 2 loss: 0.3811
Dev F1 = 0.7002
Saved best model
Epoch 3 loss: 0.3335
Dev F1 = 0.7420
Saved best model
Epoch 4 loss: 0.3107
Dev F1 = 0.7382
  Early stop counter = 1/3
Epoch 5 loss: 0.2901
Dev F1 = 0.7482
Saved best model
Epoch 6 loss: 0.2721
Dev F1 = 0.7381
  Early stop counter = 1/3
Epoch 7 loss: 0.2610
Dev F1 = 0.7444
  Early stop counter = 2/3
Epoch 8 loss: 0.2431
Dev F1 = 0.7402
  Early stop counter = 3/3
 Early stopping triggered!
 Results
Best Dev F1 = 0.7482
Test F1     = 0.7388


# **Bài 3**

In [68]:
%%writefile "/content/drive/MyDrive/DL-TH3/phoner_dataset.py"
import json
import torch
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, json_path, word2idx, label2idx):
        self.sentences = []
        self.labels = []

        # đọc JSON Lines
        with open(json_path, "r", encoding="utf-8") as f:
            data = [json.loads(line) for line in f if line.strip()]

        for sample in data:
            words = sample["words"]
            tags = sample["tags"]

            self.sentences.append([word2idx.get(w.lower(), word2idx["<unk>"]) for w in words])
            self.labels.append([label2idx[t] for t in tags])

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return {
            "tokens": self.sentences[idx],
            "labels": self.labels[idx]
        }

def collate_fn(batch, pad_idx=0):
    max_len = max(len(x["tokens"]) for x in batch)

    input_ids = []
    tag_ids = []
    mask = []

    for item in batch:
        tokens = item["tokens"]
        labels = item["labels"]

        pad_len = max_len - len(tokens)

        input_ids.append(tokens + [pad_idx] * pad_len)
        tag_ids.append(labels + [0] * pad_len)  # 0 dùng cho padding
        mask.append([1]*len(tokens) + [0]*pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(tag_ids, dtype=torch.long),
        "mask": torch.tensor(mask, dtype=torch.bool)
    }


Overwriting /content/drive/MyDrive/DL-TH3/phoner_dataset.py


In [63]:
%%writefile "/content/drive/MyDrive/DL-TH3/bilstm_crf.py"
import torch
import torch.nn as nn
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, num_labels, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 256, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=256,
            bidirectional=True,
            num_layers=5,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(256*2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, labels=None, mask=None):
        x = self.embed(input_ids)
        x, _ = self.lstm(x)
        emissions = self.fc(x)

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss

        # decode
        return self.crf.decode(emissions, mask=mask)

Overwriting /content/drive/MyDrive/DL-TH3/bilstm_crf.py


In [69]:
%%writefile "/content/drive/MyDrive/DL-TH3/train_phoner.py"
import os
import json
import torch
from torch.utils.data import DataLoader
from seqeval.metrics import f1_score, classification_report

from phoner_dataset import NERDataset, collate_fn
from bilstm_crf import BiLSTM_CRF

DATA = "/content/drive/MyDrive/DL-TH3/PhoNER_COVID19/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def read_jsonlines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_json = read_jsonlines(os.path.join(DATA, "train_word.json"))
dev_json   = read_jsonlines(os.path.join(DATA, "dev_word.json"))
test_json  = read_jsonlines(os.path.join(DATA, "test_word.json"))

words = set()
labels = set()
for dset in [train_json, dev_json, test_json]:
    for item in dset:
        words.update([w.lower() for w in item["words"]])
        labels.update(item["tags"])

word2idx = {"<pad>":0, "<unk>":1}
for w in words:
    word2idx[w] = len(word2idx)

label2idx = {label: i for i, label in enumerate(sorted(labels))}
idx2label = {i: l for l, i in label2idx.items()}

train_ds = NERDataset(os.path.join(DATA, "train_word.json"), word2idx, label2idx)
dev_ds   = NERDataset(os.path.join(DATA, "dev_word.json"), word2idx, label2idx)
test_ds  = NERDataset(os.path.join(DATA, "test_word.json"), word2idx, label2idx)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = BiLSTM_CRF(len(word2idx), len(label2idx), pad_idx=0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_f1 = 0
patience = 3
wait = 0
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        loss = model(ids, labels, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch} - Train loss: {total_loss:.4f}")

    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for batch in dev_loader:
            ids = batch["input_ids"].to(device)
            labels = batch["labels"]
            mask = batch["mask"].to(device)
            pred = model(ids, mask=mask)

            for p, g, m in zip(pred, labels, mask.cpu()):
                real_len = sum(m).item()
                preds.append([idx2label[x] for x in p[:real_len]])
                golds.append([idx2label[x.item()] for x in g[:real_len]])

    f1 = f1_score(golds, preds)
    print(f"Dev F1 = {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        wait = 0
        torch.save(model.state_dict(), "/content/drive/MyDrive/DL-TH3/best_model.pt")
        print("Model improved, saving checkpoint.")
    else:
        wait += 1
        print(f"No improvement. Wait {wait}/{patience}")

    if wait >= patience:
        print("Early stopping triggered!")
        break

print("\nTEST SET")
model.load_state_dict(torch.load("/content/drive/MyDrive/DL-TH3/best_model.pt"))
model.eval()

preds, golds = [], []
with torch.no_grad():
    for batch in test_loader:
        ids = batch["input_ids"].to(device)
        labels = batch["labels"]
        mask = batch["mask"].to(device)
        pred = model(ids, mask=mask)
        for p, g, m in zip(pred, labels, mask.cpu()):
            real_len = sum(m).item()
            preds.append([idx2label[x] for x in p[:real_len]])
            golds.append([idx2label[x.item()] for x in g[:real_len]])

print("Test F1:", f1_score(golds, preds))
print(classification_report(golds, preds))


Overwriting /content/drive/MyDrive/DL-TH3/train_phoner.py


In [70]:
!python "/content/drive/MyDrive/DL-TH3/train_phoner.py"

Epoch 0 - Train loss: 5222.9766
Dev F1 = 0.6894
Model improved, saving checkpoint.
Epoch 1 - Train loss: 1145.2735
Dev F1 = 0.8605
Model improved, saving checkpoint.
Epoch 2 - Train loss: 584.9439
Dev F1 = 0.8825
Model improved, saving checkpoint.
Epoch 3 - Train loss: 383.7245
Dev F1 = 0.8823
No improvement. Wait 1/3
Epoch 4 - Train loss: 288.7803
Dev F1 = 0.8930
Model improved, saving checkpoint.
Epoch 5 - Train loss: 202.3851
Dev F1 = 0.8984
Model improved, saving checkpoint.
Epoch 6 - Train loss: 183.1931
Dev F1 = 0.9053
Model improved, saving checkpoint.
Epoch 7 - Train loss: 180.2422
Dev F1 = 0.8931
No improvement. Wait 1/3
Epoch 8 - Train loss: 126.4020
Dev F1 = 0.8965
No improvement. Wait 2/3
Epoch 9 - Train loss: 104.0031
Dev F1 = 0.8954
No improvement. Wait 3/3
Early stopping triggered!

TEST SET
Test F1: 0.890412140024142
                     precision    recall  f1-score   support

                AGE       0.96      0.93      0.95       582
               DATE       0.96  